# Class imbalance và EDA

Dataset: Give Me Some Credit.

Mục tiêu notebook:
- Kiểm tra class distribution và vì sao accuracy dễ gây hiểu nhầm.
- Vẽ các hình EDA chính: target distribution, missing values, histogram, correlation heatmap.
- Lưu hình vào `outputs/figures/` để dùng trong report.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# -- Cấu hình style ----------------------------------------------
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
PALETTE = ['#2ecc71', '#e74c3c']   # xanh = không rủi ro, đỏ = rủi ro
RANDOM_STATE = 42

# -- Đường dẫn ----------------------------------------------------
DATA_PATH    = os.path.join('data', 'raw', 'cs-training.csv')
OUTPUT_DIR   = os.path.join('outputs', 'figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup xong.')
print(f'Đọc dữ liệu từ : {DATA_PATH}')
print(f'Lưu hình vào   : {OUTPUT_DIR}')

## 1. Đọc dữ liệu

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)   # cột đầu là index (Unnamed: 0)

print(f'Shape: {df.shape}: {df.shape[0]:,} dòng, {df.shape[1]} cột')
print()
df.head(3)

In [ ]:
# Kiểu dữ liệu & missing values tổng quan
info = pd.DataFrame({
    'dtype'      : df.dtypes,
    'non_null'   : df.notna().sum(),
    'null_count' : df.isna().sum(),
    'null_%'     : (df.isna().mean() * 100).round(2),
})
print(info.to_string())

---
## 2. Class imbalance

### 2.1 Ví dụ accuracy gây hiểu nhầm

Giả sử có 10 000 mẫu: 9 900 bình thường và 100 rủi ro.  
Một model luôn đoán 0 (không rủi ro) vẫn đạt accuracy = **99%**.  
Nhưng recall với minority class = **0%**, tức là bỏ sót toàn bộ nhóm rủi ro.

In [ ]:
# Ví dụ accuracy gây hiểu nhầm-
n_normal = 9_900
n_risk   = 100
n_total  = n_normal + n_risk

# Model luôn dự đoán class 0
acc_lazy   = n_normal / n_total
recall_lazy = 0 / n_risk          # không phát hiện được ca nào rủi ro

print('=== Ví dụ accuracy gây hiểu nhầm ===')
print(f'Tổng mẫu      : {n_total:,}')
print(f'  Bình thường : {n_normal:,} ({n_normal/n_total:.1%})')
print(f'  Rủi ro      : {n_risk:,}  ({n_risk/n_total:.1%})')
print()
print(f'Model luôn dự đoán class 0:')  
print(f'  Accuracy = {acc_lazy:.1%}   ← trông rất tốt!')
print(f'  Recall (rủi ro) = {recall_lazy:.1%}  ← không phát hiện được ca nào!')
print()
print('-> Accuracy dễ gây hiểu nhầm trên dữ liệu mất cân bằng.')

---
## 3. Hình 1 - Phân phối target

**Caption:** *Hình 1 - Phân phối nhãn target `SeriousDlqin2yrs` trong tập huấn luyện (150 000 mẫu).  
Lớp 0 (không rủi ro) chiếm ~93.3%, lớp 1 (rủi ro) chỉ ~6.7%, tỷ lệ minority:majority khoảng 1:14.*

In [ ]:
TARGET = 'SeriousDlqin2yrs'

counts = df[TARGET].value_counts().sort_index()
labels = ['Không rủi ro (0)', 'Rủi ro tín dụng (1)']
pcts   = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(7, 4.5))

bars = ax.bar(labels, counts.values, color=PALETTE, edgecolor='white', linewidth=1.2, width=0.5)

# Ghi số dòng và tỷ lệ
for bar, cnt, pct in zip(bars, counts.values, pcts.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 600,
            f'{cnt:,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Phân phối nhãn target - Give Me Some Credit', fontsize=13, pad=12)
ax.set_ylabel('Số lượng mẫu', fontsize=11)
ax.set_ylim(0, counts.max() * 1.18)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Ghi tỷ lệ mất cân bằng
ax.annotate('Tỷ lệ mất cân bằng ~ 1:14',
            xy=(0.98, 0.90), xycoords='axes fraction',
            ha='right', fontsize=10, color='#555',
            bbox=dict(boxstyle='round,pad=0.3', fc='#f9f9f9', ec='#ccc'))

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'dm_gmsc_class_distribution.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Đã lưu: {save_path}')
plt.show()

---
## 4. Hình 2 - Missing values

**Caption:** *Hình 2 - Tỷ lệ giá trị khuyết thiếu theo từng biến.  
`MonthlyIncome` có ~19.8% giá trị NULL (29 731 dòng), `NumberOfDependents` có ~2.6% (3 924 dòng).  
Các biến còn lại hầu như không thiếu dữ liệu.*

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=True)
missing = missing[missing > 0]   # chỉ giữ cột có missing

fig, ax = plt.subplots(figsize=(7, 3.5))

colors = ['#e74c3c' if v > 10 else '#f39c12' for v in missing.values]
bars   = ax.barh(missing.index, missing.values, color=colors, edgecolor='white', height=0.5)

for bar, val in zip(bars, missing.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}%', va='center', fontsize=11)

ax.set_xlim(0, missing.max() * 1.25)
ax.set_xlabel('Tỷ lệ missing (%)', fontsize=11)
ax.set_title('Tỷ lệ giá trị khuyết thiếu theo biến', fontsize=13, pad=10)

# Legend
p1 = mpatches.Patch(color='#e74c3c', label='Missing > 10%')
p2 = mpatches.Patch(color='#f39c12', label='Missing ≤ 10%')
ax.legend(handles=[p1, p2], fontsize=9, loc='lower right')

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'dm_gmsc_missing_values.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Đã lưu: {save_path}')
plt.show()

---
## 5. Hình 3 - Phân phối một số feature tài chính quan trọng

**Caption:** *Hình 3 - Phân phối của 4 biến tài chính quan trọng, phân tách theo nhãn target.  
Nhóm rủi ro (đỏ) có xu hướng tập trung ở vùng `RevolvingUtilization` cao, số lần trễ hạn nhiều hơn,  
trong khi `age` và `MonthlyIncome` phân phối khá khác nhau. Phần lớn các biến thể hiện xu hướng lệch phải, đặc biệt là `RevolvingUtilizationOfUnsecuredLines` và các biến liên quan đến lịch sử trễ hạn.*

In [ ]:
# Bỏ outlier cực đoan để hình dễ đọc (giữ 99th percentile)
def clip_col(series, upper_pct=99):
    cap = series.quantile(upper_pct / 100)
    return series.clip(upper=cap)

FEATURES_TO_PLOT = [
    ('RevolvingUtilizationOfUnsecuredLines',  'Revolving Utilization\n(% tín dụng đã dùng)'),
    ('NumberOfTime30-59DaysPastDueNotWorse',  'Số lần trễ 30-59 ngày'),
    ('age',                                   'Tuổi người vay'),
    ('MonthlyIncome',                         'Thu nhập hàng tháng (USD)'),
]

df_plot = df[[TARGET] + [f for f, _ in FEATURES_TO_PLOT]].dropna()
group0 = df_plot[df_plot[TARGET] == 0]
group1 = df_plot[df_plot[TARGET] == 1]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (col, label) in zip(axes, FEATURES_TO_PLOT):
    s0 = clip_col(group0[col])
    s1 = clip_col(group1[col])

    if col == 'NumberOfTime30-59DaysPastDueNotWorse':
        s0 = s0.clip(upper=10)
        s1 = s1.clip(upper=10)

    ax.hist(s0, bins=40, density=True, alpha=0.55, color=PALETTE[0],
            label='Không rủi ro (0)', edgecolor='none')
    ax.hist(s1, bins=40, density=True, alpha=0.65, color=PALETTE[1],
            label='Rủi ro (1)',        edgecolor='none')

    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Mật độ', fontsize=9)
    ax.legend(fontsize=8)

    if col == 'NumberOfTime30-59DaysPastDueNotWorse':
        ax.set_xlim(-0.5, 10)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'dm_gmsc_key_feature_histograms.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Đã lưu: {save_path}')
plt.show()

Các biến liên quan đến lịch sử trễ hạn cho thấy sự khác biệt phân phối đáng kể giữa hai lớp. Điều này gợi ý rằng hành vi thanh toán trong quá khứ có khả năng dự báo rủi ro tín dụng tốt hơn so với các biến nhân khẩu học như tuổi hoặc thu nhập.

---
## 6. Hình 4 - Correlation Heatmap

**Caption:** *Hình 4 - Ma trận tương quan Pearson giữa các biến trong dataset.  
Ba biến trễ hạn (`NumberOfTimes90DaysLate`, `NumberOfTime60-89Days...`, `NumberOfTime30-59Days...`) có tương quan rất cao với nhau (r ~ 0.98), nên cần đọc tương quan cẩn thận khi nhận xét EDA.  
Ba biến phản ánh lịch sử trễ hạn thanh toán (`Late30-59`, `Late60-89` và `Late90+`) có mức tương quan dương cao nhất với biến mục tiêu.*

In [ ]:
# Dùng dữ liệu đã impute median cho missing để corr không bị NaN
df_corr = df.copy()
for col in df_corr.columns:
    if df_corr[col].isna().any():
        df_corr[col].fillna(df_corr[col].median(), inplace=True)

corr = df_corr.corr()

# Rút ngắn tên cột cho dễ đọc
rename_map = {
    'SeriousDlqin2yrs'                      : 'Target',
    'RevolvingUtilizationOfUnsecuredLines'   : 'RevolvingUtil',
    'NumberOfTime30-59DaysPastDueNotWorse'   : 'Late30-59',
    'NumberOfTimes90DaysLate'                : 'Late90+',
    'NumberOfTime60-89DaysPastDueNotWorse'   : 'Late60-89',
    'NumberOfOpenCreditLinesAndLoans'        : 'OpenCredit',
    'NumberRealEstateLoansOrLines'           : 'RealEstate',
    'NumberOfDependents'                     : 'Dependents',
    'DebtRatio'                              : 'DebtRatio',
    'MonthlyIncome'                          : 'MonthlyIncome',
    'age'                                    : 'Age',
}
corr.rename(index=rename_map, columns=rename_map, inplace=True)

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # chỉ hiện tam giác dưới

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 8},
    cmap='RdYlGn', center=0,
    vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    square=True, ax=ax,
    cbar_kws={'shrink': 0.75}
)

ax.set_title('Ma trận tương quan Pearson - Give Me Some Credit', fontsize=13, pad=12)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'dm_gmsc_correlation_heatmap.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Đã lưu: {save_path}')
plt.show()

target_corr_col = rename_map.get(TARGET, TARGET)
corr[target_corr_col].sort_values(ascending=False)

---
## 7. Boxplot trễ hạn theo target

Hình này dùng để đọc rõ hơn nhóm biến trễ hạn.

In [ ]:
LATE_COLS = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
]
SHORT_NAMES = ['Trễ 30-59 ngày', 'Trễ 60-89 ngày', 'Trễ ≥ 90 ngày']

df_box = df[[TARGET] + LATE_COLS].copy()
# Clip outlier 96/98 (data anomaly đã biết)
for col in LATE_COLS:
    df_box[col] = df_box[col].clip(upper=15)

fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=False)

for ax, col, name in zip(axes, LATE_COLS, SHORT_NAMES):
    data_0 = df_box[df_box[TARGET] == 0][col]
    data_1 = df_box[df_box[TARGET] == 1][col]

    bp = ax.boxplot(
        [data_0, data_1],
        patch_artist=True,
        widths=0.45,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker='.', alpha=0.15, markersize=3)
    )
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)

    ax.set_title(name, fontsize=11)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Không rủi ro (0)', 'Rủi ro (1)'], fontsize=9)
    ax.set_ylabel('Số lần (đã clip tại 15)', fontsize=9)

fig.suptitle('Phân phối số lần trễ hạn theo nhãn target', fontsize=13, y=1.01)
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'dm_gmsc_latepayment_boxplot.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Đã lưu: {save_path}')
plt.show()

Hình cho thấy nhóm khách hàng rủi ro có số lần trễ hạn cao hơn đáng kể ở cả ba khoảng thời gian 30-59 ngày, 60-89 ngày và trên 90 ngày. Kết quả này phù hợp với trực giác nghiệp vụ rằng lịch sử thanh toán là một trong những tín hiệu quan trọng nhất của nguy cơ vỡ nợ trong tương lai.

In [ ]:
late_col = 'NumberOfTimes90DaysLate'
target_col = 'SeriousDlqin2yrs'

rate = df.groupby(late_col)[target_col].agg(['mean', 'size']).reset_index()
plot_rate = rate[rate['size'] >= 20].copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(plot_rate[late_col].astype(str), plot_rate['mean'], color='#4C78A8', edgecolor='white', linewidth=1.0)

for x, y in zip(plot_rate[late_col].astype(str), plot_rate['mean']):
    ax.text(x, y + 0.015, f'{y:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Số lần trễ hạn 90 ngày', fontsize=11)
ax.set_ylabel('Tỷ lệ default', fontsize=11)
ax.set_title('Tỷ lệ default theo số lần trễ hạn 90 ngày', fontsize=13, pad=10)
ax.set_ylim(0, max(plot_rate['mean']) * 1.12)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'dm_gmsc_default_rate_by_90dayslate.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Đã lưu: {save_path}')
plt.show()

---
## 8. Tóm tắt thống kê cơ bản (để tham khảo khi viết report)

In [ ]:
# Tỷ lệ class
vc = df[TARGET].value_counts()
print('=== Phân phối target ===')
for k, v in vc.items():
    print(f'  Class {k}: {v:,} ({v/len(df):.2%})')
ir = vc[0] / vc[1]
print(f' Imbalance Ratio (majority/minority) = {ir:.1f}')
print(f' Tỷ lệ minority:majority khoảng 1:{ir:.1f}')
print()

# Missing
print('=== Missing values ===')
miss = df.isna().sum()
miss = miss[miss > 0]
for col, n in miss.items():
    print(f'  {col}: {n:,} ({n/len(df):.2%})')
print()

# Thống kê mô tả của một số biến quan trọng
print('=== Thống kê mô tả (theo target) ===')
key_cols = ['RevolvingUtilizationOfUnsecuredLines', 'age', 'MonthlyIncome']
for col in key_cols:
    g = df.groupby(TARGET)[col].agg(['mean','median','std']).round(3)
    print(f'\n{col}:')
    print(g.to_string())

---
## 9. Kiểm tra output - Xác nhận đã lưu đủ hình

In [ ]:
expected_files = [
    'dm_gmsc_class_distribution.png',
    'dm_gmsc_missing_values.png',
    'dm_gmsc_key_feature_histograms.png',
    'dm_gmsc_correlation_heatmap.png',
    'dm_gmsc_latepayment_boxplot.png',
]

print('=== Kiểm tra file output ===')
all_ok = True
for fname in expected_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    exists = os.path.exists(fpath)
    status = 'OK' if exists else 'THIEU'
    print(f'  {status}  {fpath}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('Đủ hình cần dùng.')
else:
    print('Có file chưa được lưu. Chạy lại ô tương ứng phía trên.')